# 001 — Time Series Forecasting

Multi-step time series forecasting: moving-average baseline, SARIMA (statsmodels), and LightGBM with lag+rolling features. Covers STL decomposition, ACF/PACF, time-based split, and walk-forward evaluation.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained — no external library imports from a shared src/ package.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Data loading
feature_paths: list[str] = []          # local or gs:// URIs; empty -> synthetic
date_col: str = 'date'
target_col: str = 'value'

# Series properties
freq: str = 'D'                        # D=daily, W=weekly, M=monthly

# Forecasting
forecast_horizon: int = 30

# LightGBM feature engineering
n_lags: int = 28
rolling_windows: list[int] = [7, 14, 28]

# Optuna
optuna_n_trials: int = 20
optuna_timeout_s: int | None = None

# Train/test split
test_size: float = 0.2
random_state: int = 42

# Outputs
metrics_json_path: str = 'outputs/metrics/metrics.json'
model_output_path: str = 'outputs/models/model.lgb'
plots_dir: str = 'outputs/plots'
executed_notebook_path: str | None = None

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import polars as pl
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

optuna.logging.set_verbosity(optuna.logging.WARNING)

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

# Create output directories
for _d in [plots_dir, 'outputs/runs', 'outputs/models', 'outputs/metrics']:
    Path(_d).mkdir(parents=True, exist_ok=True)

RUN_START = datetime.now(timezone.utc)
print(f'Run started at: {RUN_START.isoformat()}')

## 1 — Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def _read_file(path: str) -> pd.DataFrame:
    """Read a single parquet or CSV file (local or gs://)."""
    p = str(path)
    if p.endswith('.parquet'):
        return pd.read_parquet(p)
    else:
        return pd.read_csv(p)


def load_series(
    paths: list[str],
    date_column: str,
    target_column: str,
    frequency: str,
) -> pd.Series:
    """Load series from files or generate synthetic data."""
    if not paths:
        # Generate a synthetic 3-year daily series
        n_days = 365 * 3
        dates = pd.date_range(start='2021-01-01', periods=n_days, freq='D')
        t = np.arange(n_days, dtype=float)
        rng = np.random.default_rng(42)
        trend = 0.05 * t
        weekly = 5.0 * np.sin(2 * np.pi * t / 7)
        yearly = 15.0 * np.sin(2 * np.pi * t / 365.25)
        noise = rng.normal(0, 2, size=n_days)
        values = 100.0 + trend + weekly + yearly + noise
        series = pd.Series(values, index=dates, name=target_column)
        print('Generated synthetic daily series (3 years, trend + seasonality + noise)')
        return series

    frames = [_read_file(p) for p in paths]
    df = pd.concat(frames, ignore_index=True)
    df[date_column] = pd.to_datetime(df[date_column])
    df = df.sort_values(date_column).set_index(date_column)
    series = df[target_column].dropna()
    print(f'Loaded {len(series):,} rows from {len(paths)} file(s)')
    return series


series = load_series(feature_paths, date_col, target_col, freq)

print(f'\nSeries info:')
print(f'  Observations : {len(series):,}')
print(f'  Date range   : {series.index.min().date()} -> {series.index.max().date()}')
print(f'  Mean         : {series.mean():.4f}')
print(f'  Std          : {series.std():.4f}')
print(f'  Min          : {series.min():.4f}')
print(f'  Max          : {series.max():.4f}')

## 2 — EDA

In [ ]:
# ---------------------------------------------------------------------------
# Plot raw series
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(series.index, series.values, linewidth=0.8, color='steelblue')
ax.set_title('Raw Time Series', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel(target_col)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.tight_layout()
_path = os.path.join(plots_dir, 'eda_series.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

In [ ]:
# ---------------------------------------------------------------------------
# STL decomposition
# ---------------------------------------------------------------------------
_period_map = {'D': 7, 'W': 52, 'M': 12}
stl_period = _period_map.get(freq, 7)

stl = STL(series, period=stl_period, robust=True)
stl_result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    ('Observed', series.values),
    ('Trend', stl_result.trend),
    ('Seasonal', stl_result.seasonal),
    ('Residual', stl_result.resid),
]
colors = ['steelblue', 'darkorange', 'seagreen', 'tomato']
for ax, (label, data), color in zip(axes, components, colors):
    ax.plot(series.index, data, linewidth=0.8, color=color)
    ax.set_ylabel(label, fontsize=10)
axes[0].set_title(f'STL Decomposition (period={stl_period})', fontsize=14)
axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.tight_layout()
_path = os.path.join(plots_dir, 'eda_stl_decomposition.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

In [ ]:
# ---------------------------------------------------------------------------
# ACF and PACF
# ---------------------------------------------------------------------------
_max_lags = min(40, len(series) // 4)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(series, lags=_max_lags, ax=axes[0], title='ACF')
plot_pacf(series, lags=_max_lags, ax=axes[1], title='PACF', method='ywm')
plt.tight_layout()
_path = os.path.join(plots_dir, 'eda_acf_pacf.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

## 3 — Train / Test Split (time-based)

In [ ]:
# ---------------------------------------------------------------------------
# Time-based train / test split
# ---------------------------------------------------------------------------
n_test = max(forecast_horizon, int(len(series) * test_size))
train_series = series.iloc[:-n_test]
test_series = series.iloc[-n_test:]

print(f'Train : {len(train_series):,} obs  ({train_series.index.min().date()} -> {train_series.index.max().date()})')
print(f'Test  : {len(test_series):,} obs  ({test_series.index.min().date()} -> {test_series.index.max().date()})')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_series.index, train_series.values, label='Train', linewidth=0.8, color='steelblue')
ax.plot(test_series.index, test_series.values, label='Test', linewidth=0.8, color='tomato')
ax.axvline(x=test_series.index[0], color='black', linestyle='--', linewidth=1, label='Split')
ax.set_title('Train / Test Split', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel(target_col)
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.tight_layout()
_path = os.path.join(plots_dir, 'train_test_split.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

## 4 — Baseline: Moving Average

In [ ]:
# ---------------------------------------------------------------------------
# Moving average baseline
# ---------------------------------------------------------------------------
ma_window = min(7, len(train_series))
ma_value = float(train_series.iloc[-ma_window:].mean())
ma_forecast = np.full(len(test_series), ma_value)

y_true = test_series.values

def _mape(actual, predicted):
    mask = actual != 0
    return float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)

ma_mae = mean_absolute_error(y_true, ma_forecast)
ma_rmse = float(np.sqrt(mean_squared_error(y_true, ma_forecast)))
ma_mape = _mape(y_true, ma_forecast)

print(f'Moving Average Baseline (window={ma_window})')
print(f'  Constant forecast value : {ma_value:.4f}')
print(f'  MAE   : {ma_mae:.4f}')
print(f'  RMSE  : {ma_rmse:.4f}')
print(f'  MAPE  : {ma_mape:.2f}%')

## 5 — SARIMA

In [ ]:
# ---------------------------------------------------------------------------
# SARIMA(1,1,1)(1,0,1)[period]
# ---------------------------------------------------------------------------
print(f'Fitting SARIMA(1,1,1)(1,0,1)[{stl_period}] ...')

sarima_model = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, stl_period),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)

sarima_forecast_result = sarima_fit.get_forecast(steps=len(test_series))
sarima_mean = sarima_forecast_result.predicted_mean.values
sarima_ci = sarima_forecast_result.conf_int().values  # shape (n, 2)

sarima_mae = mean_absolute_error(y_true, sarima_mean)
sarima_rmse = float(np.sqrt(mean_squared_error(y_true, sarima_mean)))
sarima_mape = _mape(y_true, sarima_mean)

print(f'SARIMA results:')
print(f'  MAE   : {sarima_mae:.4f}')
print(f'  RMSE  : {sarima_rmse:.4f}')
print(f'  MAPE  : {sarima_mape:.2f}%')

## 6 — LightGBM with Lag Features

In [ ]:
# ---------------------------------------------------------------------------
# Feature engineering: lags + rolling stats + calendar
# ---------------------------------------------------------------------------

def make_lag_features(
    s: pd.Series,
    lags: int,
    windows: list[int],
) -> pd.DataFrame:
    """Create lag, rolling, and calendar features from a time series."""
    df = pd.DataFrame({'y': s.values}, index=s.index)

    # Lag features (shift by 1 to avoid leakage)
    for lag in range(1, lags + 1):
        df[f'lag_{lag}'] = df['y'].shift(lag)

    # Rolling statistics
    for w in windows:
        rolled = df['y'].shift(1).rolling(window=w, min_periods=1)
        df[f'roll_mean_{w}'] = rolled.mean()
        df[f'roll_std_{w}'] = rolled.std()
        df[f'roll_min_{w}'] = rolled.min()
        df[f'roll_max_{w}'] = rolled.max()

    # Calendar features
    df['dayofweek'] = df.index.dayofweek
    df['dayofyear'] = df.index.dayofyear
    df['month'] = df.index.month
    df['quarter'] = df.index.quarter
    df['year'] = df.index.year

    df = df.dropna()
    return df


full_df = make_lag_features(series, n_lags, rolling_windows)

# Time-based split using the same test boundary
split_date = test_series.index[0]
train_df = full_df[full_df.index < split_date]
test_df = full_df[full_df.index >= split_date]

FEATURE_COLS = [c for c in full_df.columns if c != 'y']

print(f'Features     : {len(FEATURE_COLS)}')
print(f'Train rows   : {len(train_df):,}')
print(f'Test rows    : {len(test_df):,}')
print(f'Features     : {FEATURE_COLS[:8]} ...')

In [ ]:
# ---------------------------------------------------------------------------
# Optuna hyperparameter search with time-series CV
# ---------------------------------------------------------------------------
import math

X_train_all = train_df[FEATURE_COLS].values
y_train_all = train_df['y'].values


def _ts_cv_mae(params: dict, X: np.ndarray, y: np.ndarray, n_folds: int = 3) -> float:
    """3-fold sequential time-series cross-validation; returns mean MAE."""
    fold_size = len(X) // (n_folds + 1)
    maes = []
    for fold in range(n_folds):
        train_end = fold_size * (fold + 1)
        val_end = train_end + fold_size
        if val_end > len(X):
            break
        X_tr, y_tr = X[:train_end], y[:train_end]
        X_val, y_val = X[train_end:val_end], y[train_end:val_end]
        model = lgb.LGBMRegressor(**params, random_state=random_state, n_jobs=-1, verbosity=-1)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        maes.append(mean_absolute_error(y_val, preds))
    return float(np.mean(maes)) if maes else math.inf


def _objective(trial: optuna.Trial) -> float:
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    return _ts_cv_mae(params, X_train_all, y_train_all)


study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=random_state))
study.optimize(_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s, show_progress_bar=False)

best_cv_mae = study.best_value
best_params = study.best_params
print(f'Best CV MAE  : {best_cv_mae:.4f}')
print(f'Best params  : {best_params}')

In [ ]:
# ---------------------------------------------------------------------------
# Train final LightGBM model and evaluate on test set
# ---------------------------------------------------------------------------
X_test_all = test_df[FEATURE_COLS].values
y_test_all = test_df['y'].values

lgb_model = lgb.LGBMRegressor(**best_params, random_state=random_state, n_jobs=-1, verbosity=-1)
lgb_model.fit(X_train_all, y_train_all)

lgb_preds = lgb_model.predict(X_test_all)

lgb_mae = mean_absolute_error(y_test_all, lgb_preds)
lgb_rmse = float(np.sqrt(mean_squared_error(y_test_all, lgb_preds)))
lgb_mape = _mape(y_test_all, lgb_preds)
lgb_r2 = r2_score(y_test_all, lgb_preds)

# Save model
Path(model_output_path).parent.mkdir(parents=True, exist_ok=True)
lgb_model.booster_.save_model(model_output_path)

print(f'LightGBM results on test set:')
print(f'  MAE   : {lgb_mae:.4f}')
print(f'  RMSE  : {lgb_rmse:.4f}')
print(f'  MAPE  : {lgb_mape:.2f}%')
print(f'  R2    : {lgb_r2:.4f}')
print(f'  Model saved to: {model_output_path}')

## 7 — Evaluation

In [ ]:
# ---------------------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------------------
results = {
    'moving_average_baseline': {'MAE': ma_mae, 'RMSE': ma_rmse, 'MAPE': ma_mape},
    'sarima':                  {'MAE': sarima_mae, 'RMSE': sarima_rmse, 'MAPE': sarima_mape},
    'lightgbm':                {'MAE': lgb_mae, 'RMSE': lgb_rmse, 'MAPE': lgb_mape},
}

results_df = pd.DataFrame(results).T.sort_values('MAE')
best_model_name = results_df.index[0]

print('Model comparison (sorted by MAE):')
print(results_df.to_string(float_format=lambda x: f'{x:.4f}'))
print(f'\nBest model by MAE: {best_model_name}')

In [ ]:
# ---------------------------------------------------------------------------
# Forecast comparison plot
# ---------------------------------------------------------------------------
_tail = 60
train_tail = train_series.iloc[-_tail:]
test_idx = test_series.index

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_tail.index, train_tail.values, color='steelblue', linewidth=0.9, label='Train (last 60)')
ax.plot(test_idx, test_series.values, color='black', linewidth=1.2, label='Actual')
ax.plot(test_idx, ma_forecast, color='gray', linewidth=1.2, linestyle='--',
        label=f'MA Baseline (MAE={ma_mae:.2f})')
ax.plot(test_idx, sarima_mean, color='darkorange', linewidth=1.2,
        label=f'SARIMA (MAE={sarima_mae:.2f})')
ax.fill_between(test_idx, sarima_ci[:, 0], sarima_ci[:, 1],
                alpha=0.2, color='darkorange', label='SARIMA 95% CI')
ax.plot(test_idx, lgb_preds, color='seagreen', linewidth=1.2,
        label=f'LightGBM (MAE={lgb_mae:.2f})')
ax.axvline(x=test_idx[0], color='black', linestyle=':', linewidth=1)
ax.set_title('Forecast Comparison', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel(target_col)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate()
plt.tight_layout()
_path = os.path.join(plots_dir, 'forecast_comparison.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

In [ ]:
# ---------------------------------------------------------------------------
# Metric comparison bar chart
# ---------------------------------------------------------------------------
model_labels = ['MA Baseline', 'SARIMA', 'LightGBM']
mae_vals = [ma_mae, sarima_mae, lgb_mae]
rmse_vals = [ma_rmse, sarima_rmse, lgb_rmse]

x = np.arange(len(model_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, mae_vals, width, label='MAE', color='steelblue')
bars2 = ax.bar(x + width / 2, rmse_vals, width, label='RMSE', color='tomato')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_title('MAE & RMSE by Model', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.set_ylabel('Error')
ax.legend()
plt.tight_layout()
_path = os.path.join(plots_dir, 'metric_comparison.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

In [ ]:
# ---------------------------------------------------------------------------
# LightGBM residuals
# ---------------------------------------------------------------------------
lgb_residuals = y_test_all - lgb_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(test_df.index, lgb_residuals, color='seagreen', linewidth=0.9)
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_title('LightGBM Residuals over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Residual')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

axes[1].hist(lgb_residuals, bins=30, color='seagreen', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')

fig.autofmt_xdate()
plt.tight_layout()
_path = os.path.join(plots_dir, 'lgb_residuals.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

In [ ]:
# ---------------------------------------------------------------------------
# LightGBM feature importance (top 20, gain)
# ---------------------------------------------------------------------------
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': lgb_model.booster_.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1], color='steelblue')
ax.set_title('LightGBM Feature Importance (Top 20, Gain)', fontsize=13)
ax.set_xlabel('Gain')
plt.tight_layout()
_path = os.path.join(plots_dir, 'feature_importance.png')
fig.savefig(_path, bbox_inches='tight')
plt.show()
print(f'Saved: {_path}')

## 8 — Metrics JSON Output

In [ ]:
# ---------------------------------------------------------------------------
# Write metrics JSON
# ---------------------------------------------------------------------------
run_metadata = {
    'timestamp': RUN_START.isoformat(),
    'params': {
        'feature_paths': feature_paths,
        'date_col': date_col,
        'target_col': target_col,
        'freq': freq,
        'forecast_horizon': forecast_horizon,
        'n_lags': n_lags,
        'rolling_windows': rolling_windows,
        'optuna_n_trials': optuna_n_trials,
        'optuna_timeout_s': optuna_timeout_s,
        'test_size': test_size,
        'random_state': random_state,
    },
    'n_train': len(train_df),
    'n_test': len(test_df),
    'best_lgb_cv_mae': best_cv_mae,
    'best_lgb_params': best_params,
    'model_output_path': model_output_path,
}

metrics_dict = {
    'moving_average_baseline': {'mae': ma_mae, 'rmse': ma_rmse, 'mape': ma_mape},
    'sarima':                  {'mae': sarima_mae, 'rmse': sarima_rmse, 'mape': sarima_mape},
    'lightgbm':                {'mae': lgb_mae, 'rmse': lgb_rmse, 'mape': lgb_mape, 'r2': lgb_r2},
}

output_payload = {
    'run_metadata': run_metadata,
    'metrics': metrics_dict,
}

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, 'w') as f:
    json.dump(output_payload, f, indent=2)

print(f'Metrics saved to: {metrics_json_path}')

## 9 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
run_end = datetime.now(timezone.utc)
duration = (run_end - RUN_START).total_seconds()

print('=' * 62)
print('  TIME SERIES FORECASTING — RUN SUMMARY')
print('=' * 62)
print(f'  Duration        : {duration:.1f}s')
print(f'  Observations    : train={len(train_series):,}  test={len(test_series):,}')
print()
print('  Metric Results (MAE | RMSE | MAPE)')
print('  ─' * 30)
for model_name, m in results.items():
    marker = ' <-- best' if model_name == best_model_name else ''
    print(f'  {model_name:<28s}  {m["MAE"]:.3f} | {m["RMSE"]:.3f} | {m["MAPE"]:.1f}%{marker}')
print()
print(f'  LightGBM R2     : {lgb_r2:.4f}')
print(f'  Best model      : {best_model_name}')
print()
print(f'  Model path      : {model_output_path}')
print(f'  Metrics path    : {metrics_json_path}')
print(f'  Plots dir       : {plots_dir}')
if executed_notebook_path:
    print(f'  Notebook path   : {executed_notebook_path}')
print('=' * 62)